In [2]:
import enum
from pydantic import BaseModel
import pandas as pd
from vertexai.generative_models import (
    FunctionDeclaration,
    GenerationConfig,
    GenerativeModel,
    Tool,
    HarmCategory,
    HarmBlockThreshold
)
import json
import re
import pickle
# from google import genai
# import google.generativeai as genai
# from google import generativeai as genai
from google.cloud import secretmanager
import numpy as np



In [ ]:
#gsutil -m cp -r gs://sales_recommender_dev_bucket/data .

Filters

In [3]:
stage_filter = {"high": ["Biddate Set", "Construction Documents", "General Contractor Award", "Low Bids Announced", "SUBBIDS: ASAP"], "moderate": ["Construction Underway", "Post Bid"], "ignore": ["Cancelled", "Design Development", "Disqualified Lead", "Duplicate Project", "Pre-Design", "Schematic Design"]}


In [4]:
materials_df = pd.read_csv('data/relevant_materials.csv')
materials = materials_df.to_json(orient='records')

In [5]:
with open('data/boolean_filters_latest.json') as f:  # Use 'r' for reading
    product_filter_json = json.load(f)  ## Load the JSON data

Get sample ConstructCOnnect data

In [6]:
construct_connect_data = pd.read_csv("data/curated_data/sample_data.csv")

In [7]:
construct_connect_data.head()

In [8]:
PROJECT_ID = "proj-sales-recommender-dev"
LOCATION = "us-central1" 

import vertexai

vertexai.init(project=PROJECT_ID, location=LOCATION)

In [9]:
MODEL_ID = "gemini-2.0-flash-001"

model = GenerativeModel(
    MODEL_ID,
    safety_settings={
            HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
        },

)

Product filter

In [16]:
def run_prompt_boolean_filter(product, search_query, cc_project_json):
    question = f''' 
    **Objective:** Identify if the ConstructConnect project is related to the given product.

    **Instructions:**
        1. **Analyze the provided JSON data** representing ConstructConnect project to understand relevant fields and data.
        2. **Utilize the Boolean filters** corresponding to the **Product Category** to identify if the project is related to the given product by matching across multpile project fields.
        3. **Respond as YES or NO with a reason for your answer in a valid JSON as provided in the Example Output below. RETURN ONLY THE JSON RESPONSE.**

    **JSON Project Data:** 
    {cc_project_json}

    **Product Category:**
    {product}
   
    **Boolean Filters:**
    {search_query}

    **Example Output:**
    {{"ProjectID": 1006193703, "Product": {product}, "Project related to Product": "YES/NO", "Product Reasoning": reason for relation response}}
    '''

    prompt = question
    contents = [prompt]

    # Generate text using non-streaming method
    response = model.generate_content(contents)
    return response.text


In [ ]:
from tqdm import tqdm

eligible_for_material_filter = pd.DataFrame()  # Initialize an empty DataFrame

all_rows = []

for index, row in tqdm(list(construct_connect_data.iterrows())):  # Iterate through rows
    row_json_str = row.to_json()  # Convert row to dictionary
    cc_project_json_record = json.loads(row_json_str) 
    for item in tqdm(product_filter_json):
        output = run_prompt_boolean_filter(item['Filter'], item['Query'], cc_project_json_record)
        otpt = output.replace("```json", "").replace("```", "")
        try:
            output_json = json.loads(otpt)
            if (output_json["Project related to Product"]).lower() == "yes":
                new_row = row.to_dict()  # Convert row to dictionary for easier appending
                new_row['Product'] = item['Filter']
                new_row['Query'] = item['Query']
                new_row['Product_priority'] = item['Priority']
                all_rows.append(new_row)  # Append the modified row
            
        except:
            print("Error formatting: ",otpt)
            
            # Concatenate the item DataFrame with the original row
if all_rows: # Check if any rows were added to prevent errors if the list is empty
    eligible_for_material_filter = pd.DataFrame(all_rows)  # Create DataFrame from the list of dictionaries



In [ ]:
eligible_for_material_filter = eligible_for_material_filter.drop('Unnamed: 0', axis=1)

In [ ]:
data_pkl_path = 'data/curated_data/eligible_for_material_filter.pkl'
with open(data_pkl_path, 'wb') as file:
        pickle.dump(eligible_for_material_filter, file)

Material filter

In [ ]:
def run_prompt_material_filter(materials, cc_project_json):
    question = f''' 
    **Objective:** Identify if the ConstructConnect project has any of the given relevant materials.

    **Instructions:**
        1. **Analyze the provided JSON data** representing ConstructConnect project to understand relevant fields and data.
        2. **Utilize the Materials list** to identify if the project has any of the materials listed by matching across multpile project fields.
        3. **Respond as YES or NO with a list of all the materials mentioned in your answer in a valid JSON as provided in the Example Output below. RETURN ONLY THE JSON RESPONSE.**

    **JSON Project Data:** 
    {cc_project_json}

    **Materials List:**
    {materials}
   

    **Example Output:**
    {{"ProjectID": 1006193703, "Relevant materials present": YES/NO, "Materials": ["Material1","Material2", ...], "Reasoning": reason for relation response}}
    '''

    prompt = question
    contents = [prompt]

    # Generate text using non-streaming method
    response = model.generate_content(contents)
    return response.text

In [20]:
from tqdm import tqdm


eligible_for_classification = pd.DataFrame()  # Initialize an empty DataFrame
invalid_json = 0

for index, row in tqdm(eligible_for_material_filter.iterrows()):  # Iterate through rows
    row_json_str = row.to_json()  # Convert row to dictionary
    cc_project_json_record = json.loads(row_json_str) 
    #print(cc_project_json_record)
    
    output = run_prompt_material_filter(materials, cc_project_json_record)
    otpt = output.replace("```json", "").replace("```", "")
    try:
        output_json = json.loads(otpt)
        if (output_json["Relevant materials present"]).lower() == "yes":
            eligible_for_classification = pd.concat([eligible_for_classification, row.to_frame().T], ignore_index=True) # Append the row
    except:
        invalid_json += 1
            


In [ ]:
data_pkl_path = 'data/curated_data/eligible_for_classification.pkl'
with open(data_pkl_path, 'wb') as file:
        pickle.dump(eligible_for_classification, file)

Select sample for classification

In [10]:
data_pkl_path = 'data/curated_data/eligible_for_classification.pkl'

In [11]:
with open(data_pkl_path, 'rb') as file:
    eligible_for_classification = pickle.load(file)

In [12]:

eligible_for_classification = eligible_for_classification.drop(columns=['PRODUCT_PRIORITY', 'Query', 'Valuation_Bucket'], errors='ignore')
eligible_for_classification = eligible_for_classification.rename(columns={"PRODUCT": "Product"}, inplace=False)

In [13]:
unique_product_list = eligible_for_classification['Product'].unique()

In [14]:
import numpy as np


sampled_rows = []
for project_id, group in eligible_for_classification.groupby('ProjectID'):
    # Get the unique products for the current project
    products = group['Product'].unique()
    
    # Randomly select one product
    selected_product = np.random.choice(products)
    
    # Select the rows corresponding to the selected product within the project
    selected_rows = group[group['Product'] == selected_product]
    
    #Randomly select one row from the selected rows.
    sampled_rows.append(selected_rows.sample(n=1))

# Concatenate the sampled rows into a new DataFrame
sampled_eligible_for_classification = pd.concat(sampled_rows)

#Reset the index
sampled_eligible_for_classification = sampled_eligible_for_classification.reset_index(drop=True)

In [15]:
import pandas as pd

value_counts_dfs = {}
for col in ['Product']:
        counts = eligible_for_classification[col].value_counts(normalize=True)
        value_counts_dfs[col] = pd.DataFrame(counts).reset_index()
        value_counts_dfs[col].columns = [col, 'Proportion']

In [16]:

for col in ['Product']:
        counts = sampled_eligible_for_classification[col].value_counts(normalize=True)
        value_counts_dfs[f'{col}_sample'] = pd.DataFrame(counts).reset_index()
        value_counts_dfs[f'{col}_sample'].columns = [col, 'Proportion']

In [17]:
for col in ['Product']:
    # Merge the DataFrames for the full and
    comparison = value_counts_dfs[col].merge(value_counts_dfs[f'{col}_sample'], on=col, suffixes=('', '_sample'))
    comparison['Proportion_diff'] = comparison['Proportion'] - comparison['Proportion_sample']
    print(f'Comparison of {col} value counts:')
    print(comparison)
    print('\n')

    print(f'Summary statistics for the difference in proportions:')
    print(comparison['Proportion_diff'].apply(lambda x: abs(x)).describe())
    print('\n---------------------------------------------------\n')

In [102]:
pd.options.display.max_columns = None

ranking_columns = """Title
Stage
Valuation_Value
Valuation_Currency
Parameters_Parameter_Ownership
Parameters_Parameter_WorkType
Parameters_Parameter_Structures
DocumentAvailability_Plans
DocumentAvailability_Specs
DocumentAvailability_Addenda
ParentCategories_PrimaryCategoryName
ParentCategories_ParentCategory
Addresses_Address
Details_Detail_Scope
Details_Detail_Notes
Details_Detail
RSMeansMaterialDivisions_Division_Metals
RSMeansMaterialDivisions_Division_ThermalandMoistureProtection
RSMeansMaterialDivisions_Division_Openings
RSMeansMaterialDivisions_Division_Finishes
RSMeansMaterialDivisions_Division_Masonry
Details_Detail_Details
Materials_Material
Notes_Note"""

ranking_columns = ranking_columns.split("\n")


# sampled_eligible_for_classification_no_value = sampled_eligible_for_classification.drop(columns=['Valuation_Bucket', 'Valuation_Value', 'Valuation_Currency', 'Valuation_ValueType', 'Numeric_Valuation'], errors='ignore')
sampled_eligible_for_classification_no_value = sampled_eligible_for_classification[['ProjectID', 'Product'] + [col for col in sampled_eligible_for_classification.columns if col in ranking_columns]]
sampled_eligible_for_classification_no_value.head(3)

In [103]:
sampled_eligible_for_classification_no_value['Details_Detail_Scope'].head(3).values

In [104]:
[filter['Query'] for filter in product_filter_json if filter['Filter'] == sampled_eligible_for_classification_no_value['Product'].values[0]]     

Relevance classification

In [ ]:
def prompt_v1(cc_project_json, search_terms, all_searches):
    question = f'''

    **Objective:** Classify ConstructConnect projects as very high, high, moderate, low, very low, or not relevant.

        **Instructions:**

        1. **Analyze the provided JSON data:** Understand the project details, including relevant fields and data points.  The JSON data representing ConstructConnect project is provided in **Project Data**.

        2. **Utilize search terms** to identify relevant products, materials, and phrases. The search terms are privided in the **Search Terms** section.

        3. **Classify Projects:**
            * Prioritize projects based on how relevant the inputs are to the search terms.
            * When other factors are equal, prioritize higher-value projects (e.g., higher total dollar amount).

        4. **Estimate Relevancy:** Estimate the relevancy of each project based on the matching materials and total dollar amount.

        5. **Respond in valid JSON:** Return ONLY the JSON response, as shown in the example below. The JSON should contain the classification of the project as very high, high, moderate, low, or not relevant along with the reasoning as shown in the example below.

        **Input Data:**

        **Project Data:**
        {cc_project_json}
    
        **Search Terms:**
        {search_terms}

        **Example Output:**
        [
        {{"ProjectID": 1006193703, "Relevance": "Very High", "Reasoning": "Project mentions 'steel beams' and 'concrete mix,' matching our key products in the 'Structural Steel' and 'Concrete' categories. The project is in the 'Post Bid' stage, located in Illinois, and has a high value."}},
        {{"ProjectID": 1006193706, "Relevance": "Moderate", "Reasoning": "Project mentions 'roofing materials,' which is relevant to our 'Roofing' category. The project is in the 'Biddate Set' stage, location unknown (assumed outside Illinois), and value unknown (assumed low)."}},
        {{"ProjectID": 1837563703, "Relevance": "Not Relevant", "Reasoning": "No relevant products mentioned. Project is in the 'Planning' stage and located outside Illinois."}}
        ]
        '''
    prompt = question
    contents = [prompt]

    # Generate text using non-streaming method
    response = model.generate_content(contents)
    return response.text


In [106]:
from tqdm import tqdm

product_responses = []
for index, row in tqdm(list(sampled_eligible_for_classification_no_value.iterrows())):
    product = row['Product']
    row = row.drop('Product')
    search_terms = [filter['Query'] for filter in product_filter_json if filter['Filter'] == product][0]
    row_json_str = row.to_json()
    cc_project_json_record = json.loads(row_json_str)
    output = prompt_v1(row_json_str, search_terms, product)

    otpt = output.replace("```json", "").replace("```", "")  # Clean up any code block markers
    product_responses.append(otpt)

In [107]:

data=[]
for item in product_responses:
    try:
        parsed_item = json.loads(item)
        data.extend(parsed_item)
    except:
        invalid_json = item
        print(invalid_json)
        print("Invalid JSON")
        item = json.loads(invalid_json.replace('\n', '').replace('\\', ''))
        data.extend(item)
classified_projects_df = pd.DataFrame(data)

In [108]:
classified_projects_df

In [109]:
classified_projects_df['Relevance'].value_counts(normalize=True)

In [110]:
classified_projects_df['Relevance'].value_counts()


In [235]:
merged_df = pd.merge(sampled_eligible_for_classification, classified_projects_df, on=['ProjectID'], how='inner')
merged_df['Query'] = merged_df['Product'].apply(lambda x: [filter['Query'] for filter in product_filter_json if filter['Filter'] == x][0])
merged_df = merged_df[["ProjectID", "Product", "Query", "Relevance", "Reasoning"] + ranking_columns]
merged_df.head(3)

In [ ]:
import re
from ast import literal_eval

def preprocess_json_string(json_str):
    # Replace array(...) with a list
    json_str = re.sub(r"array\((\[.*?\]),\s*dtype=object\)", r"\1", json_str)
    json_str = json_str.replace("'", "\"")  # Replace single quotes with double quotes
    json_str = json_str.replace('\n', ',')  # Replace newlines with commas
    json_str = re.sub(r'\}\s*\{', '},{', json_str)  # Add commas between dictionary elements
    json_str = f'[{json_str}]' if not json_str.startswith('[') else json_str  # Ensure the string is a list

    # Handle Decimal values
    json_str = re.sub(r'Decimal\("([0-9.eE+-]+)"\)', r'\1', json_str)

    return json.loads(json_str)


# Function to format the JSON-like string into a more readable format
def format_json_string(json_str):
    try:
        json_obj = preprocess_json_string(json_str)
        formatted_str = ""
        for item in json_obj:
            name = item.get("_Name", "N/A")
            subcategories = item.get("ns0:SubCategories", {}).get("ns0:SubCategory", [])
            subcategories_str = ", ".join(subcategories)
            formatted_str += f"Category: {name}, Subcategories: {subcategories_str}\n"
        return formatted_str.strip()
    except json.JSONDecodeError:
        return json_str  # Return the original string if it cannot be parsed
    

def format_rsmeans_string(json_str):
    try:
        json_obj = preprocess_json_string(json_str)
        formatted_str = ""
        for item in json_obj:
            name = item.get("_", "N/A")
            code = item.get("_Code", "N/A")
            installation_cost = item.get("_InstallationCostValue", "N/A")
            material_cost = item.get("_MaterialCostValue", "N/A")
            total_cost = item.get("_TotalCostValue", "N/A")
            formatted_str += (f"Name: {name}, Code: {code}, Installation Cost: {installation_cost}, "
                              f"Material Cost: {material_cost}, Total Cost: {total_cost}\n")
        return formatted_str.strip()
    except json.JSONDecodeError:
        return json_str  # Return the original string if it cannot be parsed


# Apply the function to the DataFrame column
merged_df['ParentCategories_ParentCategory'] = merged_df['ParentCategories_ParentCategory'].apply(lambda x: format_json_string(x))
merged_df['Materials_Material'] = merged_df['Materials_Material'].apply(lambda x: [i['_'] for i in preprocess_json_string(x)])

for col in ['RSMeansMaterialDivisions_Division_Metals', 'RSMeansMaterialDivisions_Division_ThermalandMoistureProtection', 'RSMeansMaterialDivisions_Division_Openings', 'RSMeansMaterialDivisions_Division_Finishes', 'RSMeansMaterialDivisions_Division_Masonry']:
    merged_df[col] = merged_df[col].apply(lambda x: format_rsmeans_string(x))
    
merged_df

In [237]:
import json
import pprint

# Convert the values to JSON strings with indentation
reasoning_values = json.dumps(list(merged_df['Reasoning'].values), indent=4)

# Print the indented JSON strings
pprint.pprint(reasoning_values)

In [238]:
merged_df.to_csv("data/curated_data/sample_data_20250310.csv")